<a href="https://colab.research.google.com/github/Fahad565/safa_agent/blob/master/safa_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
!pip install -q -U google-genai "pandas<3.0.0"

import pandas as pd
print(f"✅ Libraries ready! (Pandas version: {pd.__version__})")

✅ Libraries ready! (Pandas version: 2.2.3)


In [19]:
# ═══════════════════════════════════════════════════════════════
# 1. THE SAFA PRODUCT CATALOG (Your real products!)
# ═══════════════════════════════════════════════════════════════
safa_products = {
    "Hair Growth Oil": {
        "ingredients": "Rosemary, Castor oil, Peppermint, Coconut oil base",
        "benefits": "Stimulates follicles, thickens strands, reduces breakage",
        "best_for": ["hair growth", "thinning", "breakage", "dry scalp"],
        "usage": "Massage into scalp 3x/week, leave 2+ hours or overnight"
    },
    "Organic Shampoo": {
        "ingredients": "Aloe vera, Tea tree, Coconut-derived cleansers, Hibiscus",
        "benefits": "Cleanses without stripping, soothes itch, fights dandruff",
        "best_for": ["dandruff", "itchy scalp", "oily scalp", "all hair types"],
        "usage": "Wash 2-3x/week"
    },
    "Beard Growth Oil": {
        "ingredients": "Jojoba, Argan, Cedarwood, Vitamin E",
        "benefits": "Softens beard, kills itch, fills patchy zones",
        "best_for": ["beard growth", "patchy beard", "beard itch", "dry beard"],
        "usage": "Few drops daily, massage into skin under beard"
    },
    "Natural Detergent": {
        "ingredients": "Soap nuts, Baking soda, Essential oils, Plant-based surfactants",
        "benefits": "Hypoallergenic, tough on stains, safe for babies & sensitive skin",
        "best_for": ["sensitive skin", "baby clothes", "eczema", "chemical-free home"],
        "usage": "Use like regular detergent"
    }
}

# ═══════════════════════════════════════════════════════════════
# 2. THE HARSH CHEMICAL DATABASE (For the Ingredient Checker)
# ═══════════════════════════════════════════════════════════════
harsh_chemicals = [
    ("sulfate", "Sulfates (SLS/SLES)", "Harsh foaming detergents that strip your scalp's natural oils, causing dryness and irritation."),
    ("paraben", "Parabens", "Preservatives linked to hormone disruption. Often hidden as methyl-/propylparaben."),
    ("parfum", "Synthetic Fragrance (Parfum)", "An undisclosed chemical cocktail and a top cause of skin allergies."),
    ("fragrance", "Synthetic Fragrance", "An undisclosed chemical cocktail and a top cause of skin allergies."),
    ("dimethicone", "Silicones (Dimethicone)", "Give fake smoothness but build up over time, suffocating hair and blocking moisture."),
    ("mineral oil", "Mineral Oil", "A petroleum by-product that coats hair and blocks moisture absorption."),
    ("formaldehyde", "Formaldehyde", "A preservative and known carcinogen."),
    ("triclosan", "Triclosan", "An antibacterial agent linked to hormone disruption."),
    ("phthalate", "Phthalates", "Plasticizers linked to hormone disruption, often hidden under 'fragrance'.")
]

# Show your catalog
df_products = pd.DataFrame([{"product": k, **v} for k, v in safa_products.items()])
print("🌿 THE SAFA PRODUCT CATALOG:")
display(df_products)

print(f"\n⚠️ HARSH CHEMICAL DATABASE: {len(harsh_chemicals)} chemicals tracked")
print("✅ Safa knowledge base loaded!")

🌿 THE SAFA PRODUCT CATALOG:


,product,ingredients,benefits,best_for,usage
0,Hair Growth Oil,"Rosemary, Castor oil, Peppermint, Coconut oil ...","Stimulates follicles, thickens strands, reduce...","[hair growth, thinning, breakage, dry scalp]","Massage into scalp 3x/week, leave 2+ hours or ..."
1,Organic Shampoo,"Aloe vera, Tea tree, Coconut-derived cleansers...","Cleanses without stripping, soothes itch, figh...","[dandruff, itchy scalp, oily scalp, all hair t...",Wash 2-3x/week
2,Beard Growth Oil,"Jojoba, Argan, Cedarwood, Vitamin E","Softens beard, kills itch, fills patchy zones","[beard growth, patchy beard, beard itch, dry b...","Few drops daily, massage into skin under beard"
3,Natural Detergent,"Soap nuts, Baking soda, Essential oils, Plant-...","Hypoallergenic, tough on stains, safe for babi...","[sensitive skin, baby clothes, eczema, chemica...",Use like regular detergent



⚠️ HARSH CHEMICAL DATABASE: 9 chemicals tracked
✅ Safa knowledge base loaded!


In [20]:
from google import genai
from google.colab import userdata

# 1. Option: Paste key here
API_KEY = ""

# Fallback to your Colab Secret named "GEMINI_API_KEY"
if not API_KEY:
    try:
        API_KEY = userdata.get('GEMINI_API_KEY')
    except:
        pass

if not API_KEY:
    print("⚠️ API Key not found! Please ensure you have a secret named 'GEMINI_API_KEY' in the 🔑 menu or paste it in the 'API_KEY' variable.")
else:
    try:
        client = genai.Client(api_key=API_KEY)
        # Using the specific requested model
        model_id = "gemma-4-26b-a4b-it"
        print(f"✅ Safa AI Engine connected! Model: {model_id}")
    except Exception as e:
        print(f"❌ Connection failed: {e}")

✅ Safa AI Engine connected! Model: gemma-4-26b-a4b-it


In [21]:
# ═══════════════════════════════════════════════════════════════
# TOOL 1: Product Recommender
# ═══════════════════════════════════════════════════════════════
def recommend_safa_product(concern: str) -> str:
    """Searches the Safa product catalog and returns the best-matching product(s) with benefits and usage for a customer's concern or goal (e.g., 'patchy beard', 'dandruff', 'baby clothes')."""
    concern = concern.lower()
    matches = []

    for name, info in safa_products.items():
        for keyword in info["best_for"]:
            # Match full phrase OR any distinctive single word
            if keyword in concern or any(w in concern for w in keyword.split() if len(w) > 4):
                matches.append(name)
                break

    if not matches:
        return "No exact match. Safa's core lineup: Hair Growth Oil, Organic Shampoo, Beard Growth Oil, Natural Detergent. Ask the customer to clarify their goal."

    results = []
    for name in dict.fromkeys(matches):
        info = safa_products[name]
        results.append(f"🌿 {name} | Ingredients: {info['ingredients']} | Benefits: {info['benefits']} | How to use: {info['usage']}")

    return "RECOMMENDED SAFA PRODUCTS:\n" + "\n".join(results)


# ═══════════════════════════════════════════════════════════════
# TOOL 2: 30-Day Plan Builder
# ═══════════════════════════════════════════════════════════════
def build_30day_plan(goal: str) -> str:
    """Returns a structured week-by-week 30-day routine plan using Safa products for goals like 'beard growth', 'hair growth', 'dandruff control', or 'chemical-free home'."""
    goal = goal.lower()

    plans = {
        "beard": "WEEK 1 (Days 1-7): Wash beard 2x with Safa Organic Shampoo. Apply Safa Beard Growth Oil every night. Mild itch is normal. | WEEK 2 (Days 8-14): Oil daily + 2-min massage to stimulate follicles. Itch fades. | WEEK 3 (Days 15-21): Comb daily to train direction. Keep nightly oil. | WEEK 4 (Days 22-30): Trim strays, maintain routine, take Day-30 photo to compare.",
        "hair": "WEEK 1: Scalp massage with Safa Hair Growth Oil 3x/week (leave 2+ hrs). Wash 2x/week with Safa Organic Shampoo. | WEEK 2: Same routine. Expect less breakage, calmer scalp. | WEEK 3: Add ONE overnight oil treatment. | WEEK 4: Check for baby hairs at hairline. Real growth shows week 4-8.",
        "dandruff": "WEEK 1: Wash with Safa Organic Shampoo 3x (tea tree fights flakes). Light oil on non-wash days. | WEEK 2: Flakes reduce. Wash 2-3x/week. | WEEK 3-4: Maintain 2x/week. Scalp rebalances after chemical shampoo 'purge'.",
        "home": "WEEK 1: Switch ALL laundry to Safa Natural Detergent. Wash baby/sensitive clothes first. | WEEK 2: Notice reduced skin irritation. Deep-clean bedding. | WEEK 3-4: Fully chemical-free home. Share your Safa story!"
    }

    for key, plan in plans.items():
        if key in goal:
            return f"30-DAY SAFA PLAN ({key.upper()}):\n{plan}"
    return "No matching plan. Available: beard growth, hair growth, dandruff control, chemical-free home. Ask the customer to pick one."


# ═══════════════════════════════════════════════════════════════
# TOOL 3: Ingredient Safety Checker (The Transparency Feature!)
# ═══════════════════════════════════════════════════════════════
def check_ingredient_safety(ingredients_text: str) -> str:
    """Checks a pasted product ingredient list against Safa's harsh-chemical database. Returns flagged chemicals with plain-language explanations, or confirms the list is clean."""
    text = ingredients_text.lower()
    flagged = []
    seen = set()

    for key, name, explanation in harsh_chemicals:
        if key in text and name not in seen:
            seen.add(name)
            flagged.append(f"⚠️ {name}: {explanation}")

    if flagged:
        return "HARSH CHEMICALS FOUND:\n" + "\n".join(flagged) + "\n\nA Safa natural alternative exists for this product type - recommend it."
    return "✅ No harsh chemicals detected. This list looks clean!"


# ═══════════════════════════════════════════════════════════════
# TOOL 4: Journey Troubleshooter (Retention & Trust Builder)
# ═══════════════════════════════════════════════════════════════
def troubleshoot_symptom(week: int, symptom: str) -> str:
    """Gives coaching advice for common symptoms customers report during their 30-day natural journey, based on the week number and symptom (e.g., week=2, symptom='itch')."""
    symptom = symptom.lower()

    if "itch" in symptom:
        return "Itch in weeks 1-2 is NORMAL - your skin is rebalancing after chemical products. Apply Safa oil at night, don't scratch. Fades by week 3."
    if "no growth" in symptom or "not growing" in symptom:
        return "Hair grows ~1.25cm/month - too slow to see daily. Take weekly photos in same lighting. Visible change at week 4-8. Stay consistent!"
    if "greasy" in symptom or "oily" in symptom:
        return "You're using too much oil. Reduce to 3-5 drops, massage fully in. Natural oils absorb best on slightly damp hair/beard."
    if "flake" in symptom or "purge" in symptom:
        return "This is the 'transition purge' - your scalp is detoxing from silicones/sulfates in old products. Lasts 1-2 weeks. Keep using Safa Organic Shampoo."
    return "Stay consistent with your Safa routine, hydrate, and sleep well. If it persists beyond week 3, consult a professional."


# ═══════════════════════════════════════════════════════════════
# QUICK MANUAL TEST (See the tools working BEFORE the AI uses them)
# ═══════════════════════════════════════════════════════════════
print(" TEST 1 - Product Recommender:")
print(recommend_safa_product("my beard is patchy and itches"))
print("\n TEST 2 - Ingredient Checker (a typical commercial shampoo):")
print(check_ingredient_safety("Aqua, Sodium Laureth Sulfate, Cocamidopropyl Betaine, Parfum, Methylparaben, Dimethicone"))

 TEST 1 - Product Recommender:
RECOMMENDED SAFA PRODUCTS:
🌿 Beard Growth Oil | Ingredients: Jojoba, Argan, Cedarwood, Vitamin E | Benefits: Softens beard, kills itch, fills patchy zones | How to use: Few drops daily, massage into skin under beard

 TEST 2 - Ingredient Checker (a typical commercial shampoo):
HARSH CHEMICALS FOUND:
⚠️ Sulfates (SLS/SLES): Harsh foaming detergents that strip your scalp's natural oils, causing dryness and irritation.
⚠️ Parabens: Preservatives linked to hormone disruption. Often hidden as methyl-/propylparaben.
⚠️ Synthetic Fragrance (Parfum): An undisclosed chemical cocktail and a top cause of skin allergies.
⚠️ Silicones (Dimethicone): Give fake smoothness but build up over time, suffocating hair and blocking moisture.

A Safa natural alternative exists for this product type - recommend it.


In [23]:
# 1. Initialize the model WITH your 4 tools AND the brand personality using the correct Client syntax
model_config = {
    "tools": [recommend_safa_product, build_30day_plan, check_ingredient_safety, troubleshoot_symptom],
    "system_instruction": """You are the Safa Glow-Up Coach, the friendly and knowledgeable AI salesperson for Safa, an organic beauty & home care brand.

YOUR RULES:
1. ALWAYS use your tools to look up products, plans, or ingredient analysis BEFORE answering. Never guess.
2. Be warm, encouraging, and HONEST. Never overpromise (e.g., say visible growth takes 4-8 weeks).
3. When you flag harsh chemicals, ALWAYS follow up by recommending the matching Safa alternative.
4. Keep replies concise, friendly, and easy to read. Use a few emojis.
5. End conversations by making the customer feel supported on their natural journey."""
}

# 2. Start the chat with automatic function calling
# Note: model_id 'gemini-1.5-flash' is used as 'gemma-4-26b-a4b-it' is invalid
chat = client.chats.create(
    model="gemma-4-26b-a4b-it",
    config=model_config
)

# 3. Simple function to run a customer conversation
def talk_to_coach(customer_message: str):
    print(f"🧑 CUSTOMER: {customer_message}")
    print("-" * 70)
    response = chat.send_message(customer_message)
    print(f"🌿 SAFA COACH:\n{response.text}")
    print("=" * 70)

print("✅ Safa Glow-Up Coach is LIVE!")

✅ Safa Glow-Up Coach is LIVE!


In [24]:
# CONVERSATION 1: The frustrated beard customer
talk_to_coach("My beard is so patchy and it itches like crazy. Can you help me?")

# CONVERSATION 2: The suspicious customer checking their old shampoo
talk_to_coach("Should I be worried? Here are the ingredients of my current shampoo: Aqua, Sodium Laureth Sulfate, Cocamidopropyl Betaine, Parfum, Methylparaben, Dimethicone.")

# CONVERSATION 3: The week-2 check-in (the coach remembers the customer!)
talk_to_coach("It's week 2 of my plan and my beard still itches. Is this normal?")

🧑 CUSTOMER: My beard is so patchy and it itches like crazy. Can you help me?
----------------------------------------------------------------------
🌿 SAFA COACH:
Oh, I hear you! Dealing with a patchy beard and that constant itch can be so frustrating. 😩 But don't worry, we can definitely work on this together!

Based on what you're experiencing, I highly recommend our **🌿 Beard Growth Oil**. It’s specifically formulated to tackle both of your concerns:

*   **For the patches:** It uses nutrient-rich Jojoba and Argan oils to nourish your hair follicles and help fill in those thin spots.
*   **For the itch:** The Cedarwood and Vitamin E help soothe the skin underneath, calming that irritation almost immediately.

**How to use it:** Just apply a few drops daily and massage it gently into the skin beneath your beard. 💆‍♂️

Keep in mind that natural growth takes a little patience—you'll likely start seeing real changes in about 4 to 8 weeks. Consistency is your best friend here!

Would you 

### 🚀 Step 1: Create the Streamlit App File
This code creates a file named `safa_app.py`. In Streamlit, you must use `st.secrets` instead of Colab's `userdata` to handle your API key securely.

In [2]:
%%writefile safa_app.py
import streamlit as st
from google import genai

# Page Config
st.set_page_config(page_title="Safa Glow-Up Coach", page_icon="🌿")
st.title("🌿 Safa Glow-Up Coach")
st.markdown("Welcome to your natural beauty journey!")

# Initialize Client (Using Streamlit Secrets for production)
# In Streamlit Cloud, add GEMINI_API_KEY to the 'Secrets' menu
api_key = st.secrets["GEMINI_API_KEY"]
client = genai.Client(api_key=api_key)

# --- Define your tools (recommend_safa_product, etc. go here) ---
# [Copy the functions from cell FMKk1GUxNVrJ here in your local file]

# Chat Logic
if "messages" not in st.session_state:
    st.session_state.messages = []

for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

if prompt := st.chat_input("Ask about your hair or beard journey..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    # Generate AI Response
    # (In a real app, you would initialize the chat session here)
    response = "[AI Response would appear here using chat.send_message]"

    with st.chat_message("assistant", avatar="🌿"):
        st.markdown(response)
    st.session_state.messages.append({"role": "assistant", "content": response})

Writing safa_app.py


### 📦 Step 3: Create the Requirements File
Run this cell to create the `requirements.txt` file needed for GitHub and Streamlit.

In [3]:
%%writefile requirements.txt
google-genai
streamlit
pandas<3.0.0

Writing requirements.txt


### 🛠️ Final Steps for Website Integration
1. **Push to GitHub**: Upload `safa_app.py` and `requirements.txt` to a repo.
2. **Deploy**: Go to Streamlit Cloud, connect the repo, and set your `GEMINI_API_KEY` in **App Settings > Secrets** using this format:
```toml
GEMINI_API_KEY = "your-api-key-here"
```
3. **Embed**: Use the iframe code to put the coach on your homepage!

### 🌍 Step 2: How to Embed in your Future Website
Once your Streamlit app is deployed, use this HTML code in your website's code to show the coach on your store page:

```html
<iframe
  src="https://your-app-name.streamlit.app/?embed=true"
  style="height: 600px; width: 100%; border: none;"
></iframe>
```